In [1]:
%load_ext autoreload
%autoreload 2

import os
import random
import numpy as np
import pandas as pd
import data
import utils

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from finetune import CNNModelTrainer, TransModelTrainer
from model import ResNet50Model, MobileNetV2, ViT16

import torch

/home/kuniko/anaconda3/lib/python3.11/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


In [2]:
# to reproduce
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms = True
os.environ['PYTHONHASHSEED'] = str(seed)

In [3]:
label_to_class={0: 'airplane',
             1: 'automobile',
             2: 'bird',
             3: 'cat',
             4: 'deer',
             5: 'dog',
             6: 'frog',
             7: 'horse',
             8: 'ship',
             9: 'truck'}

In [4]:
def get_dataset(root):
    random_state = 42
    test_size = 0.2
    
    save_filepath = f'{root}train.csv'
    df = pd.read_csv(save_filepath)
    df_real = df[df['rf'] == 'REAL'].copy()
    df_fake = df[df['rf'] == 'FAKE'].copy()
    
    df_train_r, df_valid_r = train_test_split(df_real, test_size=test_size, random_state=random_state, shuffle=True)
    df_train_r['image'] = df_train_r['filepath'].apply(os.path.basename)
    df_valid_r['image'] = df_valid_r['filepath'].apply(os.path.basename)
    
    df_train_f, df_valid_f = train_test_split(df_fake, test_size=test_size, random_state=random_state, shuffle=True)
    df_train_f['image'] = df_train_f['filepath'].apply(os.path.basename)
    df_valid_f['image'] = df_valid_f['filepath'].apply(os.path.basename)
    
    save_filepath = f'{root}test.csv'
    df_test = pd.read_csv(save_filepath)
    df_test_r = df_test[df_test['rf'] == 'REAL'].copy()
    df_test_f = df_test[df_test['rf'] == 'FAKE'].copy()
    df_test_r['image'] = df_test_r['filepath'].apply(os.path.basename)
    df_test_f['image'] = df_test_f['filepath'].apply(os.path.basename)

    return df_train_r, df_valid_r, df_test_r, df_train_f, df_valid_f, df_test_f

In [14]:
def get_dataloaders(df_train_r, df_valid_r, df_test_r, df_train_f, df_valid_f, df_test_f, batch_size=32, img_size=(32, 32)):

    num_class = len(label_to_class)
    label_encoder = LabelEncoder()
    label_encoder.fit(df_train_r["label"])
    train_loader_r = data.get_dataloader(df_train_r, img_size, batch_size, label_encoder, train=True)
    valid_loader_r = data.get_dataloader(df_valid_r, img_size, batch_size, label_encoder, train=False)
    test_loader_r = data.get_dataloader(df_test_r, img_size, batch_size, label_encoder, train=False)
    train_loader_f = data.get_dataloader(df_train_f, img_size, batch_size, label_encoder, train=True)
    valid_loader_f = data.get_dataloader(df_valid_f, img_size, batch_size, label_encoder, train=False)
    test_loader_f = data.get_dataloader(df_test_f, img_size, batch_size, label_encoder, train=False)

    return train_loader_r, valid_loader_r, test_loader_r, train_loader_f, valid_loader_f, test_loader_f

In [16]:
def run_training(model_class, **kwargs):

    num_class = kwargs.get('num_class')
    num_epochs = kwargs.get('epochs')
    lr = kwargs.get('lr')
    model_name = kwargs.get('model_name')
    root = kwargs.get('root')
    batch_size = kwargs.get('batch_size')
    img_size = kwargs.get('img_size')

    # Data
    df_train_r, df_valid_r, df_test_r, df_train_f, df_valid_f, df_test_f = get_dataset(root)
    train_loader_r, valid_loader_r, test_loader_r, train_loader_f, valid_loader_f, test_loader_f = get_dataloaders(
        df_train_r, df_valid_r, df_test_r, df_train_f, df_valid_f, df_test_f, batch_size, img_size)

    def preparation(model_save_directory):
        # Preparation
        utils.create_directory(model_save_directory)
        utils.delete_subfolders(model_save_directory)

    def real_testing(root, model_name, setting, trainer, model, test_loader, df_test):
        display = True
        out_filepath = f'{root}results/{model_name}_{setting}_real_report.csv'
        
        preds = trainer.evaluate(model, test_loader, display, out_filepath)
        df_test['preds'] = preds
        df_test.to_csv(f'{root}results/{model_name}_{setting}_real.csv', index=False)

    def fake_testing(root, model_name, setting, trainer, model, test_loader, df_test):
        display = True
        out_filepath = f'{root}results/{model_name}_{setting}_fake_report.csv'
        
        preds = trainer.evaluate(model, test_loader, display, out_filepath)
        df_test['preds'] = preds
        df_test.to_csv(f'{root}results/{model_name}_{setting}_fake.csv', index=False)


    # Real Setting Preparation
    setting = 'real'
    model_save_directory = f"{root}{model_name}/{setting}/"
    preparation(model_save_directory)
    
    # Training
    model = model_class(num_class)
    trainer = TransModelTrainer() if model_name == 'ViT16' else CNNModelTrainer()
    best_val_file = trainer.fit(model, train_loader_r, valid_loader_r, model_save_directory, epochs=num_epochs, lr=lr)

    model = model_class(num_class)
    model.load_state_dict(torch.load(best_val_file))
    
    # Real
    real_testing(root, model_name, setting, trainer, model, test_loader_r, df_test_r)

    # Fake
    fake_testing(root, model_name, setting, trainer, model, test_loader_f, df_test_f)

    # Fake Setting Preparation
    setting = 'fake'
    model_save_directory = f"{root}{model_name}/{setting}/"
    preparation(model_save_directory)
    
    # Training
    model = model_class(num_class)
    trainer = TransModelTrainer() if model_name == 'ViT16' else CNNModelTrainer()
    best_val_file = trainer.fit(model, train_loader_f, valid_loader_f, model_save_directory, epochs=num_epochs, lr=lr)

    model = model_class(num_class)
    model.load_state_dict(torch.load(best_val_file))
    
    # Real
    real_testing(root, model_name, setting, trainer, model, test_loader_r, df_test_r)

    # Fake
    fake_testing(root, model_name, setting, trainer, model, test_loader_f, df_test_f)

    print('Real vs Fake Completed.')

# CIFAKE1

In [8]:
root = "../../dataset/CIFAKE/cifake1/"
num_class = len(label_to_class)
model_name = 'MobileNetV2'
num_epochs = 100
lr = 1e-2
batch_size = 32
img_size = (32, 32)

settings = {
    'num_class': num_class, 'epochs': num_epochs, 'lr':lr, 'model_name': model_name, 'root': root, 'batch_size': batch_size, 'img_size': img_size}

run_training(MobileNetV2, **settings)

../../dataset/CIFAKE/cifake1/MobileNetV2/real/ aready exist.
Sub folders of ../../dataset/CIFAKE/cifake1/MobileNetV2/real/ was deleted.
Epoch: 0 | Val Acc: 0.5294 | Loss: 1.3066 | F1: 0.5214
Epoch: 1 | Val Acc: 0.5789 | Loss: 1.1529 | F1: 0.5636
Epoch: 2 | Val Acc: 0.6199 | Loss: 1.0830 | F1: 0.6105
Epoch: 3 | Val Acc: 0.6444 | Loss: 1.0128 | F1: 0.6367
Epoch: 4 | Val Acc: 0.6655 | Loss: 0.9715 | F1: 0.6585
Epoch: 5 | Val Acc: 0.6742 | Loss: 0.9333 | F1: 0.6683
Epoch: 6 | Val Acc: 0.6633 | Loss: 0.9559 | F1: 0.6562
Epoch: 7 | Val Acc: 0.6833 | Loss: 0.9380 | F1: 0.6798
Epoch: 8 | Val Acc: 0.6785 | Loss: 0.9017 | F1: 0.6690
Epoch: 9 | Val Acc: 0.6978 | Loss: 0.8670 | F1: 0.6963
Epoch: 10 | Val Acc: 0.6986 | Loss: 0.8702 | F1: 0.6938
Epoch: 11 | Val Acc: 0.6998 | Loss: 0.8464 | F1: 0.6975
Epoch: 12 | Val Acc: 0.7186 | Loss: 0.8156 | F1: 0.7163
Epoch: 13 | Val Acc: 0.7003 | Loss: 0.8566 | F1: 0.6974
Epoch: 14 | Val Acc: 0.7269 | Loss: 0.7995 | F1: 0.7220
Epoch: 15 | Val Acc: 0.7214 | Loss

In [9]:
num_class = len(label_to_class)
model_name = 'ResNet50Model'
num_epochs = 100
lr = 1e-2

settings = {
    'num_class': num_class, 'epochs': num_epochs, 'lr':lr, 'model_name': model_name, 'root': root, 'batch_size': batch_size, 'img_size': img_size}
run_training(ResNet50Model, **settings)

../../dataset/CIFAKE/cifake1/ResNet50Model/real/ created.
Sub folders of ../../dataset/CIFAKE/cifake1/ResNet50Model/real/ was deleted.
Epoch: 0 | Val Acc: 0.3655 | Loss: 1.7650 | F1: 0.3605
Epoch: 1 | Val Acc: 0.5157 | Loss: 1.3628 | F1: 0.5061
Epoch: 2 | Val Acc: 0.6098 | Loss: 1.1227 | F1: 0.6021
Epoch: 3 | Val Acc: 0.6449 | Loss: 1.0194 | F1: 0.6341
Epoch: 4 | Val Acc: 0.6822 | Loss: 0.9167 | F1: 0.6779
Epoch: 5 | Val Acc: 0.7044 | Loss: 0.8554 | F1: 0.6996
Epoch: 6 | Val Acc: 0.7188 | Loss: 0.8270 | F1: 0.7127
Epoch: 7 | Val Acc: 0.7307 | Loss: 0.7733 | F1: 0.7283
Epoch: 8 | Val Acc: 0.7405 | Loss: 0.7555 | F1: 0.7366
Epoch: 9 | Val Acc: 0.7494 | Loss: 0.7204 | F1: 0.7459
Epoch: 10 | Val Acc: 0.7542 | Loss: 0.7165 | F1: 0.7492
Epoch: 11 | Val Acc: 0.7693 | Loss: 0.6635 | F1: 0.7672
Epoch: 12 | Val Acc: 0.7699 | Loss: 0.6662 | F1: 0.7664
Epoch: 13 | Val Acc: 0.7645 | Loss: 0.6917 | F1: 0.7632
Epoch: 14 | Val Acc: 0.7775 | Loss: 0.6416 | F1: 0.7757
Epoch: 15 | Val Acc: 0.7786 | Loss:

In [18]:
num_class = len(label_to_class)
model_name = 'ViT16'
num_epochs = 100
lr = 1e-5
batch_size =64
img_size = (224, 224)

settings = {
    'num_class': num_class, 'epochs': num_epochs, 'lr':lr, 'model_name': model_name, 'root': root, 'batch_size': batch_size, 'img_size': img_size}
run_training(ViT16, **settings)

../../dataset/CIFAKE/cifake1/ViT16/real/ aready exist.
Sub folders of ../../dataset/CIFAKE/cifake1/ViT16/real/ was deleted.
Epoch: 0 | Val Acc: 0.7206 | Loss: 1.0766 | F1: 0.7153
Epoch: 1 | Val Acc: 0.8521 | Loss: 0.5008 | F1: 0.8520
Epoch: 2 | Val Acc: 0.8730 | Loss: 0.3973 | F1: 0.8746
Epoch: 3 | Val Acc: 0.8995 | Loss: 0.3096 | F1: 0.8993
Epoch: 4 | Val Acc: 0.9132 | Loss: 0.2713 | F1: 0.9137
Epoch: 5 | Val Acc: 0.9150 | Loss: 0.2585 | F1: 0.9151
Epoch: 6 | Val Acc: 0.9223 | Loss: 0.2365 | F1: 0.9226
Epoch: 7 | Val Acc: 0.9231 | Loss: 0.2249 | F1: 0.9232
Epoch: 8 | Val Acc: 0.9294 | Loss: 0.2083 | F1: 0.9295
Epoch: 9 | Val Acc: 0.9270 | Loss: 0.2117 | F1: 0.9274
Epoch: 10 | Val Acc: 0.9305 | Loss: 0.2193 | F1: 0.9302
Epoch: 11 | Val Acc: 0.9364 | Loss: 0.1977 | F1: 0.9367
Epoch: 12 | Val Acc: 0.9327 | Loss: 0.2112 | F1: 0.9328
Epoch: 13 | Val Acc: 0.9392 | Loss: 0.1930 | F1: 0.9392
Epoch: 14 | Val Acc: 0.9351 | Loss: 0.2057 | F1: 0.9351
Epoch: 15 | Val Acc: 0.9354 | Loss: 0.2132 | F

# CIFAKE2

In [19]:
root = "../../dataset/CIFAKE/cifake2/"
num_class = len(label_to_class)
model_name = 'MobileNetV2'
num_epochs = 100
lr = 1e-2
batch_size = 32
img_size = (32, 32)

settings = {
    'num_class': num_class, 'epochs': num_epochs, 'lr':lr, 'model_name': model_name, 'root': root, 'batch_size': batch_size, 'img_size': img_size}

run_training(MobileNetV2, **settings)

../../dataset/CIFAKE/cifake2/MobileNetV2/real/ created.
Sub folders of ../../dataset/CIFAKE/cifake2/MobileNetV2/real/ was deleted.
Epoch: 0 | Val Acc: 0.5380 | Loss: 1.2847 | F1: 0.5261
Epoch: 1 | Val Acc: 0.5952 | Loss: 1.1285 | F1: 0.5866
Epoch: 2 | Val Acc: 0.6222 | Loss: 1.0798 | F1: 0.6173
Epoch: 3 | Val Acc: 0.6592 | Loss: 0.9909 | F1: 0.6576
Epoch: 4 | Val Acc: 0.6451 | Loss: 1.0135 | F1: 0.6362
Epoch: 5 | Val Acc: 0.6603 | Loss: 0.9646 | F1: 0.6544
Epoch: 6 | Val Acc: 0.6872 | Loss: 0.9180 | F1: 0.6805
Epoch: 7 | Val Acc: 0.6958 | Loss: 0.8830 | F1: 0.6926
Epoch: 8 | Val Acc: 0.6843 | Loss: 0.9112 | F1: 0.6778
Epoch: 9 | Val Acc: 0.6670 | Loss: 0.9495 | F1: 0.6602
Epoch: 10 | Val Acc: 0.6997 | Loss: 0.8646 | F1: 0.6941
Epoch: 11 | Val Acc: 0.6994 | Loss: 0.8596 | F1: 0.6943
Epoch: 12 | Val Acc: 0.7063 | Loss: 0.8357 | F1: 0.6996
Epoch: 13 | Val Acc: 0.7128 | Loss: 0.8186 | F1: 0.7119
Epoch: 14 | Val Acc: 0.7105 | Loss: 0.8451 | F1: 0.7067
Epoch: 15 | Val Acc: 0.7064 | Loss: 0.8

In [20]:
num_class = len(label_to_class)
model_name = 'ResNet50Model'
num_epochs = 100
lr = 1e-2

settings = {
    'num_class': num_class, 'epochs': num_epochs, 'lr':lr, 'model_name': model_name, 'root': root, 'batch_size': batch_size, 'img_size': img_size}
run_training(ResNet50Model, **settings)

../../dataset/CIFAKE/cifake2/ResNet50Model/real/ created.
Sub folders of ../../dataset/CIFAKE/cifake2/ResNet50Model/real/ was deleted.
Epoch: 0 | Val Acc: 0.4175 | Loss: 1.6696 | F1: 0.4008
Epoch: 1 | Val Acc: 0.5530 | Loss: 1.2876 | F1: 0.5369
Epoch: 2 | Val Acc: 0.6148 | Loss: 1.0928 | F1: 0.6033
Epoch: 3 | Val Acc: 0.6576 | Loss: 0.9834 | F1: 0.6500
Epoch: 4 | Val Acc: 0.6791 | Loss: 0.9079 | F1: 0.6702
Epoch: 5 | Val Acc: 0.7061 | Loss: 0.8402 | F1: 0.6997
Epoch: 6 | Val Acc: 0.7135 | Loss: 0.8153 | F1: 0.7058
Epoch: 7 | Val Acc: 0.7375 | Loss: 0.7513 | F1: 0.7338
Epoch: 8 | Val Acc: 0.7378 | Loss: 0.7620 | F1: 0.7327
Epoch: 9 | Val Acc: 0.7440 | Loss: 0.7400 | F1: 0.7382
Epoch: 10 | Val Acc: 0.7515 | Loss: 0.7197 | F1: 0.7466
Epoch: 11 | Val Acc: 0.7662 | Loss: 0.6870 | F1: 0.7620
Epoch: 12 | Val Acc: 0.7693 | Loss: 0.6749 | F1: 0.7665
Epoch: 13 | Val Acc: 0.7797 | Loss: 0.6466 | F1: 0.7772
Epoch: 14 | Val Acc: 0.7822 | Loss: 0.6416 | F1: 0.7791
Epoch: 15 | Val Acc: 0.7797 | Loss:

In [21]:
num_class = len(label_to_class)
model_name = 'ViT16'
num_epochs = 100
lr = 1e-5
batch_size = 64
img_size = (224, 224)

settings = {
    'num_class': num_class, 'epochs': num_epochs, 'lr':lr, 'model_name': model_name, 'root': root, 'batch_size': batch_size, 'img_size': img_size}
run_training(ViT16, **settings)

../../dataset/CIFAKE/cifake2/ViT16/real/ created.
Sub folders of ../../dataset/CIFAKE/cifake2/ViT16/real/ was deleted.
Epoch: 0 | Val Acc: 0.6554 | Loss: 1.1962 | F1: 0.6399
Epoch: 1 | Val Acc: 0.8487 | Loss: 0.5171 | F1: 0.8491
Epoch: 2 | Val Acc: 0.8795 | Loss: 0.3843 | F1: 0.8794
Epoch: 3 | Val Acc: 0.8959 | Loss: 0.3168 | F1: 0.8968
Epoch: 4 | Val Acc: 0.9101 | Loss: 0.2780 | F1: 0.9108
Epoch: 5 | Val Acc: 0.9130 | Loss: 0.2665 | F1: 0.9128
Epoch: 6 | Val Acc: 0.9167 | Loss: 0.2424 | F1: 0.9171
Epoch: 7 | Val Acc: 0.9256 | Loss: 0.2249 | F1: 0.9259
Epoch: 8 | Val Acc: 0.9275 | Loss: 0.2233 | F1: 0.9276
Epoch: 9 | Val Acc: 0.9289 | Loss: 0.2182 | F1: 0.9291
Epoch: 10 | Val Acc: 0.9287 | Loss: 0.2136 | F1: 0.9289
Epoch: 11 | Val Acc: 0.9299 | Loss: 0.2162 | F1: 0.9301
Epoch: 12 | Val Acc: 0.9338 | Loss: 0.2033 | F1: 0.9341
Epoch: 13 | Val Acc: 0.9288 | Loss: 0.2213 | F1: 0.9291
Epoch: 14 | Val Acc: 0.9308 | Loss: 0.2230 | F1: 0.9313
Epoch: 15 | Val Acc: 0.9371 | Loss: 0.2061 | F1: 0.